# Clase 9 — TensorFlow en Google Colaboratory, GPU/TPU y migración a TensorFlow 2.0

Curso **Fundamentos de TensorFlow desde cero** — DIH DATAlife

## Objetivos de aprendizaje

- Sacar partido a los entornos de Google Colaboratory para entrenar modelos sin instalación local
- Configurar y usar GPU (Graphics Processing Unit) y TPU (Tensor Processing Unit) para acelerar el entrenamiento
- Identificar los cambios clave al migrar código de TensorFlow 1.x a 2.x

Este notebook se ejecuta en Google Colab con TensorFlow 2.x. Algunas celdas están marcadas explícitamente como opcionales porque requieren activar GPU o TPU desde el menú **Entorno de ejecución > Cambiar tipo de entorno de ejecución**.

## 1. Qué es Google Colaboratory

Colab es un cuaderno Jupyter que corre en un entorno de ejecución (runtime) remoto de Google, con Python y TensorFlow (TF) ya instalados. No ejecutas nada en tu ordenador: tu navegador solo muestra el resultado de lo que corre en la nube.

Cosas que puedes hacer directamente desde una celda de código:

- Ejecutar comandos de terminal anteponiendo `!` (por ejemplo `!pip install tensorflow==2.15.0`)
- Montar tu Google Drive para leer y guardar archivos de forma permanente
- Subir archivos sueltos directamente a la sesión (se pierden al cerrarla)

Vamos a comprobar la versión de TensorFlow instalada por defecto en el entorno.

In [ ]:
import tensorflow as tf

print("Versión de TensorFlow:", tf.__version__)
print("Ejecución eager activada:", tf.executing_eagerly())

### Montar Google Drive (opcional, requiere permisos de tu cuenta)

Esta celda solo tiene sentido dentro de Colab, con tu propia cuenta de Google. Te pedirá autorización la primera vez. Una vez montado, cualquier archivo de tu Drive queda accesible en `/content/drive/MyDrive/...`.

In [ ]:
# Descomenta estas líneas si estás en Colab y quieres montar tu Drive
# from google.colab import drive
# drive.mount('/content/drive')
# print("Drive montado. Ejemplo de ruta: /content/drive/MyDrive/mi_carpeta/mis_datos.csv")

### Subir un archivo suelto (opcional, requiere Colab)

Útil cuando solo necesitas un archivo de forma puntual y no quieres pasar por Drive. Abre un selector de archivos en tu navegador.

In [ ]:
# Descomenta estas líneas si estás en Colab y quieres subir un archivo
# from google.colab import files
# subidos = files.upload()
# print("Archivos subidos:", list(subidos.keys()))

## 2. Comprobar qué dispositivo tienes disponible: CPU, GPU o TPU

`tf.config.list_physical_devices(...)` te dice qué hardware ve realmente TensorFlow en tu sesión actual. Esta celda funciona en cualquier entorno de ejecución, sin necesidad de activar nada especial: si no hay GPU ni TPU, simplemente verás listas vacías para esos dos tipos.

In [ ]:
print("CPU (Central Processing Unit):", tf.config.list_physical_devices('CPU'))
print("GPU (Graphics Processing Unit):", tf.config.list_physical_devices('GPU'))
print("TPU (Tensor Processing Unit):", tf.config.list_physical_devices('TPU'))

**Para ver una GPU en la lista anterior**: en el menú de Colab ve a **Entorno de ejecución > Cambiar tipo de entorno de ejecución** y selecciona GPU como acelerador de hardware. La sesión se reiniciará y, al volver a ejecutar la celda de arriba, deberías ver un dispositivo GPU listado.

La GPU y la TPU aceleran el entrenamiento porque paralelizan las multiplicaciones de matrices, la operación que domina el cálculo de una red neuronal. Una CPU tiene pocos núcleos muy versátiles; una GPU tiene cientos o miles de núcleos simples pensados para repetir la misma operación sobre muchos datos a la vez; una TPU es un chip que Google diseñó específicamente para ese tipo de cálculo, y por eso suele ser aún más rápida que una GPU en modelos grandes.

## 3. Usar la GPU en el entrenamiento

Con GPU activada, TensorFlow la usa automáticamente sin que cambies nada en tu código de Keras. Si quieres forzar una parte del cálculo a un dispositivo concreto, usas el gestor de contexto `tf.device(...)`. La siguiente celda funciona en cualquier entorno: si no hay GPU, sencillamente entrena en CPU sin dar error.

In [ ]:
import time

# Datos pequeños de ejemplo: no hace falta un dataset real para ver el patrón
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0

def crear_modelo():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

dispositivo = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print("Entrenando en:", dispositivo)

with tf.device(dispositivo):
    modelo = crear_modelo()
    modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    inicio = time.time()
    modelo.fit(x_train, y_train, epochs=1, batch_size=128, verbose=0)
    print(f"Tiempo de entrenamiento: {time.time() - inicio:.2f} segundos")

## 4. Usar una TPU: idea simplificada del ejemplo de Shakespeare

El notebook original *Predict Shakespeare with Cloud TPUs and Keras* entrena un modelo de lenguaje (dos capas LSTM) sobre el texto completo de Shakespeare para predecir el siguiente carácter, usando una TPU en Colab. Aquí vemos la misma idea simplificada: construir el modelo, crear una estrategia de distribución y entrenar dentro de su `scope()`.

**Esta celda requiere un entorno de ejecución con TPU activada** (Entorno de ejecución > Cambiar tipo de entorno de ejecución > TPU) y solo funciona dentro de Colab, porque depende de la variable de entorno `COLAB_TPU_ADDR` que Colab define automáticamente. Si no tienes TPU activada, la celda no se ejecuta: pasa a la celda siguiente, que simula el mismo patrón sin necesitar hardware real.

In [ ]:
import os

# --- CELDA QUE REQUIERE TPU REAL EN COLAB ---
if 'COLAB_TPU_ADDR' in os.environ:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(
        tpu='grpc://' + os.environ['COLAB_TPU_ADDR']
    )
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.experimental.TPUStrategy(resolver)
    print("TPU conectada. Dispositivos lógicos:", tf.config.list_logical_devices('TPU'))
else:
    print("No hay TPU activada en este entorno: activa TPU en el menú Entorno de ejecución")
    strategy = tf.distribute.get_strategy()  # estrategia "normal", sin distribución

La celda siguiente sí se ejecuta siempre, con o sin TPU: si detectamos TPU usamos `TPUStrategy`, y si no, usamos la estrategia por defecto de TensorFlow. El código de `fit()` es idéntico en ambos casos: eso es justo lo que hace útil a `tf.distribute.Strategy`, no tienes que reescribir tu ciclo de entrenamiento para adaptarlo al hardware.

In [ ]:
# Modelo LSTM muy reducido, solo para ilustrar el patrón de distribución
# (en el notebook original, EMBEDDING_DIM=512 y se entrena sobre el texto
# completo de Shakespeare durante 10 épocas; aquí lo simplificamos a un
# vocabulario y una secuencia diminutos para que corra en segundos en CPU)

VOCAB_SIZE = 256   # un carácter ASCII por token, como en el notebook original
EMBEDDING_DIM = 32 # reducido respecto al original (512) para ir rápido sin TPU
SEQ_LEN = 20

def lstm_model():
    entrada = tf.keras.Input(shape=(SEQ_LEN,), dtype=tf.int32)
    x = tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(entrada)
    x = tf.keras.layers.LSTM(EMBEDDING_DIM, return_sequences=True)(x)
    salida = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax')
    )(x)
    return tf.keras.Model(inputs=entrada, outputs=salida)

with strategy.scope():
    modelo_texto = lstm_model()
    modelo_texto.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')

# Datos aleatorios solo para comprobar que el ciclo de entrenamiento funciona
import numpy as np
x_ejemplo = np.random.randint(0, VOCAB_SIZE, size=(64, SEQ_LEN))
y_ejemplo = np.random.randint(0, VOCAB_SIZE, size=(64, SEQ_LEN, 1))

modelo_texto.fit(x_ejemplo, y_ejemplo, epochs=1, batch_size=16, verbose=1)
print("Entrenamiento de ejemplo completado (con o sin TPU, el código es el mismo)")

## 5. Diferencias de sintaxis: TensorFlow 1.x frente a 2.x

El cambio más importante entre versiones es la ejecución. TensorFlow 1.x construía un grafo de operaciones y necesitabas una `tf.Session()` para evaluarlo. TensorFlow 2.x ejecuta cada operación de forma inmediata (eager execution), como Python normal.

In [ ]:
# Estilo TensorFlow 1.x, usando el módulo de compatibilidad tensorflow.compat.v1
# Construimos las operaciones dentro de un grafo explícito (en lugar de con
# tf1.disable_eager_execution(), que afectaría a todo el notebook a partir de
# aquí) para poder comparar ambos estilos sin romper las celdas siguientes.
import tensorflow.compat.v1 as tf1

grafo_v1 = tf.Graph()
with grafo_v1.as_default():
    a1 = tf1.constant(2)
    b1 = tf1.constant(3)
    c1 = tf1.add(a1, b1)

with tf1.Session(graph=grafo_v1) as sess:
    resultado = sess.run(c1)
    print("Resultado (estilo TF 1.x, con Session):", resultado)

In [ ]:
# Estilo TensorFlow 2.x: sin sesión, sin placeholders, sin inicializar variables a mano
a2 = tf.constant(2)
b2 = tf.constant(3)
c2 = tf.add(a2, b2)
print("Resultado (estilo TF 2.x, eager):", c2.numpy())

# Una variable está lista para usarse en el mismo momento en que la creas
peso = tf.Variable(0.5, name='peso')
print("Variable lista sin inicializador aparte:", peso.numpy())

Otras diferencias que conviene recordar:

- TensorFlow 1.x usaba `tf.placeholder()` para reservar un hueco y le pasaba los datos reales mediante `feed_dict` dentro de una sesión; TensorFlow 2.x no tiene placeholders, los datos se pasan directamente a la función o al modelo
- TensorFlow 1.x requería `tf.global_variables_initializer()` antes de usar cualquier `tf.Variable`; en TensorFlow 2.x no hace falta ningún paso de inicialización
- TensorFlow 2.x integra Keras (`tf.keras`) como la API de alto nivel recomendada por defecto, en lugar de APIs de bajo nivel como `tf.layers`
- La herramienta de línea de comandos `tf_upgrade_v2` convierte automáticamente buena parte de un script de TensorFlow 1.x a 2.x y genera un informe de los cambios que no pudo resolver sola

## Resumen

- Colab te da un entorno de ejecución remoto con TensorFlow preinstalado; puedes montar Drive o subir archivos sueltos según necesites persistencia o no
- `tf.config.list_physical_devices(...)` confirma qué hardware ve TensorFlow: CPU siempre, GPU o TPU solo si las activaste en el menú del entorno de ejecución
- Con GPU, el código no cambia: TensorFlow la usa automáticamente. Con TPU, necesitas una `TPUStrategy` y definir el modelo dentro de su `scope()`
- TensorFlow 2.x elimina sesiones y placeholders, no requiere inicializar variables a mano, e integra Keras como API principal por defecto

Continúa con el notebook de práctica de esta clase para medir tú mismo la diferencia de tiempo entre CPU y GPU, y para reescribir un fragmento de código de estilo TensorFlow 1.x a TensorFlow 2.x.